# Bronze Layer — Managed Delta Tables with Auto Loader Ingestion

## Design Decisions
- **Managed Delta tables** (not external) so Unity Catalog owns both metadata and data lifecycle.
- **Auto Loader (`cloudFiles`)** for incremental, scalable file ingestion — only new files are processed on each run.
- **Schema inference + schema evolution** enabled so new JSON fields are handled automatically.
- **NOT NULL constraints** applied at the Bronze layer as the first data quality gate.
- **Partitioned by `ingestion_date`** for query pruning and efficient time-travel.

> Prerequisite: Run `0.config.ipynb` first (or let the Workflow chain handle it via `%run`).

In [0]:
%run ./0.config

In [0]:
# ── Step 1: Ensure catalog / schema exist ────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA} MANAGED LOCATION '{BRONZE_PATH}'")

In [0]:
# ── Step 2: Create target managed Delta tables (DDL with NOT NULL constraints) ─
# BIGINT used for all integer fields — Spark's JSON reader infers all integers
# as Long (64-bit) by default. Declaring INT (32-bit) causes a type conflict
# when mergeSchema=true tries to reconcile INT vs LONG at write time.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(BRONZE_SCHEMA, 'drivers')} (
    driverId       BIGINT    NOT NULL,
    driverRef      STRING    NOT NULL,
    number         BIGINT,
    code           STRING,
    name           STRUCT<forename: STRING, surname: STRING>,
    dob            STRING,
    nationality    STRING,
    url            STRING,
    ingestion_date DATE      NOT NULL,
    source_file    STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'bronze'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(BRONZE_SCHEMA, 'results')} (
    resultId        BIGINT  NOT NULL,
    raceId          BIGINT  NOT NULL,
    driverId        BIGINT  NOT NULL,
    constructorId   BIGINT  NOT NULL,
    number          BIGINT,
    grid            BIGINT,
    position        BIGINT,
    positionText    STRING,
    positionOrder   BIGINT,
    points          DOUBLE,
    laps            BIGINT,
    time            STRING,
    milliseconds    BIGINT,
    fastestLap      BIGINT,
    rank            BIGINT,
    fastestLapTime  STRING,
    fastestLapSpeed STRING,
    statusId        BIGINT,
    ingestion_date  DATE    NOT NULL,
    source_file     STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'bronze'
)
""")

In [0]:
# ── Step 3: Batch ingestion of drivers.json into managed Delta ───────────────
# For single-file sources, batch read + overwrite is simpler and more
# reliable than Auto Loader (which is designed for directory-based streams).

from pyspark.sql import functions as F

drivers_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{BRONZE_PATH}drivers.json")
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.lit(f"{BRONZE_PATH}drivers.json"))
)

(
    drivers_raw.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(fq(BRONZE_SCHEMA, "drivers"))
)

print(f"drivers loaded: {spark.table(fq(BRONZE_SCHEMA, 'drivers')).count():,} rows")

In [0]:
# ── Step 4: Batch ingestion of results.json into managed Delta ───────────────

results_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{BRONZE_PATH}results.json")
    .withColumn("ingestion_date", F.current_date())
    .withColumn("source_file",    F.lit(f"{BRONZE_PATH}results.json"))
)

(
    results_raw.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(fq(BRONZE_SCHEMA, "results"))
)

print(f"results loaded: {spark.table(fq(BRONZE_SCHEMA, 'results')).count():,} rows")

In [0]:
# ── Step 5: Data Quality Validation — row counts & NOT NULL checks ────────────

def validate_bronze(schema: str, table: str, pk_col: str):
    full_name = fq(schema, table)
    df = spark.table(full_name)
    total      = df.count()
    null_pk    = df.filter(F.col(pk_col).isNull()).count()
    null_date  = df.filter(F.col("ingestion_date").isNull()).count()

    assert total   > 0,  f"[DQ FAIL] {full_name}: table is empty!"
    assert null_pk == 0, f"[DQ FAIL] {full_name}: {null_pk} NULL values in primary key '{pk_col}'"
    assert null_date == 0, f"[DQ FAIL] {full_name}: {null_date} NULL values in ingestion_date"

    print(f"[DQ PASS] {full_name}: {total:,} rows | 0 NULL PKs | 0 NULL dates")

validate_bronze(BRONZE_SCHEMA, "drivers", "driverId")
validate_bronze(BRONZE_SCHEMA, "results", "resultId")